STEP 1 : DATA GENERATION

In [13]:
import requests

urls = [
    "https://raw.githubusercontent.com/ageron/handson-ml3/main/README.md",
    "https://raw.githubusercontent.com/pytorch/pytorch/main/README.md",
    "https://raw.githubusercontent.com/openai/openai-cookbook/main/README.md",
    "https://raw.githubusercontent.com/langchain-ai/langchain/main/README.md",
    "https://raw.githubusercontent.com/fastapi/fastapi/master/README.md"
]

docs = []

for url in urls:
    res = requests.get(url)

    docs.append({
        "title": url.split("/")[-1],
        "description": "GitHub markdown doc",
        "content": res.text
    })

print(len(docs))
print(docs[0]["content"][:300])

5
Machine Learning Notebooks, 3rd edition

This project aims at teaching you the fundamentals of Machine Learning in
python. It contains the example code and solutions to the exercises in the third edition of my O'Reilly book [Hands-on Machine Learning with Scikit-Lea


In [2]:
import re
def split_sections(text):
    sections = re.split(r"\n## ", text)
    return sections

In [3]:
section_chunks = []

for doc in docs:
    sections = split_sections(doc["content"])

    for i, sec in enumerate(sections):
        if len(sec.strip()) > 200:
            section_chunks.append({
                "title": doc["title"],
                "chunk_id": i,
                "chunk": sec
            })

print(len(section_chunks))
print(section_chunks[0]["chunk"][:300])

22
Machine Learning Notebooks, 3rd edition

This project aims at teaching you the fundamentals of Machine Learning in
python. It contains the example code and solutions to the exercises in the third edition of my O'Reilly book [Hands-on Machine Learning with Scikit-Lea


1 - TEXT SEARCH

STEP 1 - AI documentation for Search

In [4]:
docs_for_search = []

for c in section_chunks:
    docs_for_search.append({
        "chunk": c["chunk"],
        "title": c["title"],
        "description": "AI documentation",
        "filename": c["title"]
    })

STEP 2 — CREATING AN INDEX

In [5]:
from minsearch import Index

index = Index(
    text_fields=["chunk", "title", "description", "filename"],
    keyword_fields=[]
)

index.fit(docs_for_search)

STEP 3 — SEARCH

In [6]:
query = "What is machine learning?"

results = index.search(query)

for r in results[:3]:
    print("TITLE:", r["title"])
    print(r["chunk"][:300])
    print("------")

TITLE: README.md
Machine Learning Notebooks, 3rd edition

This project aims at teaching you the fundamentals of Machine Learning in
python. It contains the example code and solutions to the exercises in the third edition of my O'Reilly book [Hands-on Machine Learning with Scikit-Lea
------
TITLE: README.md
Resources

* [PyTorch.org](https://pytorch.org/)
* [PyTorch Tutorials](https://pytorch.org/tutorials/)
* [PyTorch Examples](https://github.com/pytorch/examples)
* [PyTorch Models](https://pytorch.org/hub/)
* [Intro to Deep Learning with PyTorch from Udacity](https://www.udacity.com/course/deep-learn
------
TITLE: README.md
More About PyTorch

[Learn the basics of PyTorch](https://pytorch.org/tutorials/beginner/basics/intro.html)

At a granular level, PyTorch is a library that consists of the following components:

| Component | Description |
| ---- | --- |
| [**torch**](https://pytorch.org/docs/stable/torch.html) | A 
------


2 - VECTOR SEARCH

STEP 1 — Creating the Model

In [7]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer('multi-qa-distilbert-cos-v1')

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

STEP 2 — Create the embed

In [8]:
from tqdm.auto import tqdm
import numpy as np

embeddings = []

for d in tqdm(docs_for_search):
    v = embedding_model.encode(d["chunk"])
    embeddings.append(v)

embeddings = np.array(embeddings)

  0%|          | 0/22 [00:00<?, ?it/s]

STEP 3 — Setting up VectorSearch index

In [9]:
from minsearch import VectorSearch

vindex = VectorSearch()
vindex.fit(embeddings, docs_for_search)

In [10]:
query = "What is machine learning?"

q = embedding_model.encode(query)

results = vindex.search(q)

for r in results[:3]:
    print(r["title"])
    print(r["chunk"][:300])
    print("------")

README.md
Machine Learning Notebooks, 3rd edition

This project aims at teaching you the fundamentals of Machine Learning in
python. It contains the example code and solutions to the exercises in the third edition of my O'Reilly book [Hands-on Machine Learning with Scikit-Lea
------
README.md
More About PyTorch

[Learn the basics of PyTorch](https://pytorch.org/tutorials/beginner/basics/intro.html)

At a granular level, PyTorch is a library that consists of the following components:

| Component | Description |
| ---- | --- |
| [**torch**](https://pytorch.org/docs/stable/torch.html) | A 
------
README.md
Resources

* [PyTorch.org](https://pytorch.org/)
* [PyTorch Tutorials](https://pytorch.org/tutorials/)
* [PyTorch Examples](https://github.com/pytorch/examples)
* [PyTorch Models](https://pytorch.org/hub/)
* [Intro to Deep Learning with PyTorch from Udacity](https://www.udacity.com/course/deep-learn
------


3 - HYBRID SEARCH

In [11]:
query = "What is machine learning?"
text_results = index.search(query, num_results=5)

q = embedding_model.encode(query)
vector_results = vindex.search(q, num_results=5)

final_results = text_results + vector_results


for r in final_results[:5]:
    print(r["title"])
    print(r["chunk"][:300])
    print("------")

README.md
Machine Learning Notebooks, 3rd edition

This project aims at teaching you the fundamentals of Machine Learning in
python. It contains the example code and solutions to the exercises in the third edition of my O'Reilly book [Hands-on Machine Learning with Scikit-Lea
------
README.md
Resources

* [PyTorch.org](https://pytorch.org/)
* [PyTorch Tutorials](https://pytorch.org/tutorials/)
* [PyTorch Examples](https://github.com/pytorch/examples)
* [PyTorch Models](https://pytorch.org/hub/)
* [Intro to Deep Learning with PyTorch from Udacity](https://www.udacity.com/course/deep-learn
------
README.md
More About PyTorch

[Learn the basics of PyTorch](https://pytorch.org/tutorials/beginner/basics/intro.html)

At a granular level, PyTorch is a library that consists of the following components:

| Component | Description |
| ---- | --- |
| [**torch**](https://pytorch.org/docs/stable/torch.html) | A 
------
README.md
Opinions

"_[...] I'm using **FastAPI** a ton these days. [...] I'm ac

4 - Putting this together

In [12]:
def text_search(query):
    return index.search(query, num_results=5)

def vector_search(query):
    q = embedding_model.encode(query)
    return vindex.search(q, num_results=5)

def hybrid_search(query):
    text_results = text_search(query)
    vector_results = vector_search(query)

    seen_titles = set()
    combined_results = []

    for result in text_results + vector_results:
        if result["title"] not in seen_titles:
            seen_titles.add(result["title"])
            combined_results.append(result)

    return combined_results

results = hybrid_search("What is machine learning?")

for r in results[:5]:
    print(r["title"])
    print(r["chunk"][:300])
    print("------")

README.md
Machine Learning Notebooks, 3rd edition

This project aims at teaching you the fundamentals of Machine Learning in
python. It contains the example code and solutions to the exercises in the third edition of my O'Reilly book [Hands-on Machine Learning with Scikit-Lea
------
